# ME 313 · Lab 6.2 — A heat-exchanger surrogate
**Module 6 companion (Internal Flow & LMTD).**

Sizing or rating an exchanger means solving the LMTD/energy-balance relations for every operating point. 
A **surrogate model** learns the input→output map once and then predicts in microseconds — the workhorse of design sweeps, controllers, and digital twins. 
Here the 'truth' is the constant-$T_s$ tube result $T_{m,o}=T_s-(T_s-T_{m,i})e^{-hA_s/(\dot m c_p)}$; you train a surrogate and **validate it against that physics**.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

### 1. Generate operating-condition data
Inputs: mass flow $\dot m$, inlet temp $T_{m,i}$, wall temp $T_s$, and $hA_s$. Output: outlet temp $T_{m,o}$.


In [ ]:
rng = np.random.default_rng(2); cp = 4180.
def outlet(mdot, Tmi, Ts, hAs):
    return Ts - (Ts-Tmi)*np.exp(-hAs/(mdot*cp))

N = 1200
mdot = rng.uniform(0.05, 1.0, N)
Tmi  = rng.uniform(10, 40, N)
Ts   = rng.uniform(80, 120, N)
hAs  = rng.uniform(200, 3000, N)
y = outlet(mdot, Tmi, Ts, hAs)
X = np.column_stack([mdot, Tmi, Ts, hAs])
print('dataset:', X.shape)

### 2. Train the surrogate


In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
sc = StandardScaler().fit(Xtr)
net = MLPRegressor(hidden_layer_sizes=(64,64), max_iter=2000, random_state=0)
net.fit(sc.transform(Xtr), ytr)
pred = net.predict(sc.transform(Xte))
print(f'R2 = {r2_score(yte, pred):.4f}   max abs err = {np.max(np.abs(pred-yte)):.2f} C')

### 3. Validate against the physics


In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(yte, pred, s=8, alpha=.5)
lims=[yte.min(), yte.max()]; plt.plot(lims, lims, 'k--')
plt.xlabel('true outlet T (C)'); plt.ylabel('surrogate T (C)')
plt.title('Surrogate vs. exact LMTD/energy balance'); plt.grid(alpha=.3); plt.show()

### Your turn
1. Time a sweep of 10,000 points through the exact equation vs. the surrogate — quantify the speed-up.
2. Use the surrogate inside a check: *what $\dot m$ keeps $T_{m,o}$ below a limit?* Verify the answer against the exact relation.
3. Retrain with only 100 points. How does accuracy degrade? This is the data-cost question every surrogate faces.
4. **AI companion:** have an LLM propose the exchanger relation, then confirm its exponential form matches the data-generating function above.
